# 🎵 Demucs Custom - Быстрая проверка на 7s Sample

Этот ноутбук запускает быстрое обучение на семпле для проверки,
что весь пайплайн работает. После успешной проверки можно
перейти на полный датасет.

**Главное отличие от v1:** пакет `musdb` качает **7-секундный сэмпл**,
а не полный MUSDB18 на 10 ГБ. Это официальный тестовый набор sigsep —
на нём уже можно проверить весь пайплайн от данных до сохранения модели.

**Инструкция:** нажимай ▶️ на каждой ячейке сверху вниз.

In [ ]:
# Подключение Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive подключен!")
print("Твои веса будут сохраняться в: MyDrive/demucs_custom/")

In [ ]:
# Клонирование проекта с GitHub
#
# Команды запускаем через subprocess, а не через «!».
# Такой код одинаково работает и в Colab, и в обычном Jupyter/VS Code.
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/IgorCim/demucs_custom.git"
PROJECT_DIR = "/content/demucs_custom"

if os.path.isdir(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)

subprocess.run(["git", "clone", "--depth", "1", REPO_URL, PROJECT_DIR], check=True)
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

print("✅ Код клонирован. Текущая папка:", os.getcwd())
print("Файлы проекта:", sorted(os.listdir('.')))

In [ ]:
# Установка зависимостей
print("⏳ Ставлю зависимости, это 2-4 минуты...")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "demucs", "torchaudio", "librosa", "soundfile", "musdb"], check=True)

import torch
import torchaudio
import demucs
import librosa
import soundfile

print("✅ Все зависимости установлены!")
print("torch     :", torch.__version__)
print("torchaudio:", torchaudio.__version__)
print("demucs    :", demucs.__version__)

# Проверяем ffmpeg — без него не распаковать стемы из .stem.mp4
if subprocess.run(["which", "ffmpeg"], capture_output=True).returncode != 0:
    subprocess.run(["apt-get", "install", "-y", "ffmpeg"], check=False)
ffmpeg_ok = subprocess.run(["ffmpeg", "-version"], capture_output=True).returncode == 0
print("ffmpeg    :", "✅ есть" if ffmpeg_ok else "❌ нет")
if not ffmpeg_ok:
    raise RuntimeError("ffmpeg не найден — без него ячейка 7 не сможет распаковать стемы")

In [ ]:
# Проверка GPU
import torch

print("=" * 50)
print("ПРОВЕРКА ОБОРУДОВАНИЯ")
print("=" * 50)
if torch.cuda.is_available():
    print(f"✅ GPU доступен: {torch.cuda.get_device_name(0)}")
    print(f"✅ Память GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print("Можно запускать обучение!")
else:
    print("❌ GPU НЕ доступен!")
    print("Что делать:")
    print("1. Меню Runtime → Change runtime type")
    print("2. Hardware accelerator → GPU T4")
    print("3. Save → перезапусти все ячейки")
    raise RuntimeError("Без GPU обучение не имеет смысла. Сделай шаги выше и перезапусти.")

In [ ]:
# Настройка путей
from pathlib import Path

drive_models_path = '/content/drive/MyDrive/demucs_custom/models'
Path(drive_models_path).mkdir(parents=True, exist_ok=True)

# Датасет держим на ЛОКАЛЬНОМ диске Colab, а не на Drive.
# Даже 140 МБ через FUSE-монтирование тормозят обучение —
# каждый батч будет ждать сеть диска.
SAMPLE_DIR = "/content/musdb18"      # сырое хранилище 7s сэмпла
OUT_ROOT = "/content/custom_dataset"  # формат нашего проекта
MODEL_PATH = f'{drive_models_path}/quick_test.pth'

print(f"✅ Модель:  {MODEL_PATH}")
print(f"✅ Сэмпл:   {SAMPLE_DIR}")
print(f"✅ Датасет: {OUT_ROOT}")

In [ ]:
# Скачивание 7s Sample Dataset (140 МБ, 1-2 минуты)
#
# ВАЖНО, что внутри: это НЕ wav-файлы. В архиве лежат 144 файла
# вида train/"Название трека".stem.mp4, где каждый mp4 — контейнер
# сразу с 5 аудиопотоками (AAC): mixture, drums, bass, other, vocals.
#
# Поэтому искать «*.wav» здесь бесполезно — их нет вообще.
import glob

import musdb

if os.path.isdir(os.path.join(SAMPLE_DIR, "train")):
    print("✅ Сэмпл уже скачан, пропускаем загрузку")
else:
    print("⏳ Скачивание 7s Sample Dataset (~140 МБ)...")
    musdb.DB(root=SAMPLE_DIR, download=True)
    print("✅ Скачано!")

stems_mp4 = sorted(glob.glob(os.path.join(SAMPLE_DIR, "train", "*.stem.mp4")))
print(f"\nТреков в train: {len(stems_mp4)}")
if stems_mp4:
    print("Примеры:")
    for path in stems_mp4[:3]:
        print("   ", os.path.basename(path))

In [ ]:
# Распаковка .stem.mp4 в формат проекта
#
# data_prep.py ждёт:
#     custom_dataset/train/track_001/{mixture,vocals,drums,bass,other}.wav
#     custom_dataset/valid/track_101/...
#
# Порядок потоков внутри .stem.mp4 задан в mus.yaml (demucs):
#     0 = mixture, 1 = drums, 2 = bass, 3 = other, 4 = vocals
#
# Каждый трек — всего 6.8 секунды, поэтому segment_length в ячейке 10
# уменьшен до 6 секунд, иначе модель получит куски с тишиной.
import glob
import time

import numpy as np
import soundfile as sf

STREAM_ORDER = ["mixture", "drums", "bass", "other", "vocals"]
SOURCES = ["vocals", "drums", "bass", "other"]
TARGET_SR = 44100
VALID_RATIO = 0.1
MIN_SECONDS = 4.0        # треки короче — мусор, их пропускаем

def extract_stems(mp4_path, out_dir):
    """Распаковывает 5 потоков из .stem.mp4 в отдельные wav."""
    os.makedirs(out_dir, exist_ok=True)
    for index, name in enumerate(STREAM_ORDER):
        out_path = os.path.join(out_dir, f"{name}.wav")
        # -map 0:a:N — обязательно по одному потоку: wav-муксер не
        # умеет писать несколько аудиодорожек в один файл.
        result = subprocess.run(
            ["ffmpeg", "-y", "-v", "error", "-i", mp4_path,
             "-map", f"0:a:{index}", "-ac", "2", "-ar", str(TARGET_SR),
             "-c:a", "pcm_s16le", out_path],
            capture_output=True, text=True, encoding="utf-8", errors="replace")
        if result.returncode != 0:
            return False
        if not os.path.isfile(out_path):
            return False
    return True

def _read(path):
    audio, _ = sf.read(path, always_2d=True, dtype="float32")
    return np.ascontiguousarray(audio.T)

if os.path.isdir(os.path.join(OUT_ROOT, "train")) and os.path.isdir(os.path.join(OUT_ROOT, "valid")):
    print("✅ Датасет уже распакован, пропускаем")
else:
    os.makedirs(os.path.join(OUT_ROOT, "train"), exist_ok=True)
    os.makedirs(os.path.join(OUT_ROOT, "valid"), exist_ok=True)

    n_valid = max(1, int(len(stems_mp4) * VALID_RATIO))
    print(f"Треков: {len(stems_mp4)} → train {len(stems_mp4) - n_valid}, valid {n_valid}")

    started = time.time()
    done = 0
    failed = 0
    for index, mp4_path in enumerate(stems_mp4):
        is_valid = index >= len(stems_mp4) - n_valid
        split = "valid" if is_valid else "train"
        out_dir = os.path.join(OUT_ROOT, split, f"track_{index + 1:03d}")

        if not extract_stems(mp4_path, out_dir):
            print(f"  ⚠️  не распаковался: {os.path.basename(mp4_path)}")
            failed += 1
            continue

        try:
            vocals = _read(os.path.join(out_dir, "vocals.wav"))
        except Exception as exc:
            print(f"  ⚠️  не читается: {os.path.basename(mp4_path)} ({type(exc).__name__})")
            failed += 1
            shutil.rmtree(out_dir, ignore_errors=True)
            continue

        if vocals.shape[1] < MIN_SECONDS * TARGET_SR:
            print(f"  ⚠️  пропускаю (слишком короткий): {os.path.basename(mp4_path)}")
            shutil.rmtree(out_dir, ignore_errors=True)
            failed += 1
            continue

        done += 1
        if done % 20 == 0:
            print(f"  ... готово {done}/{len(stems_mp4)} ({time.time() - started:.0f} сек)")

    print(f"\n✅ Распаковано: {done} треков, пропущено: {failed}")

counts = {}
for split in ("train", "valid"):
    path = os.path.join(OUT_ROOT, split)
    tracks = sorted(glob.glob(os.path.join(path, "track_*")))
    counts[split] = len(tracks)
    print(f"{split}: {counts[split]} треков")
    if tracks:
        print(f"  пример: {sorted(os.listdir(tracks[0]))}")

if counts["train"] == 0 or counts["valid"] == 0:
    raise RuntimeError(
        f"Датасет неполный: train={counts['train']}, valid={counts['valid']}.\n"
        "Обучение не запустится — нужны обе выборки."
    )

print("\n--- проверка одной дорожки ---")
first = sorted(glob.glob(os.path.join(OUT_ROOT, "train", "track_*")))[0]
mix = _read(os.path.join(first, "mixture.wav"))
sums = sum(_read(os.path.join(first, f"{n}.wav")) for n in SOURCES)
print("mixture shape:", mix.shape)
print("max расхождение mixture и суммы дорожек: %.4f" % np.abs(mix - sums).max())

In [ ]:
# Резервный датасет: синтетические треки
#
# Нужен только если сэмпл не распаковался (нет ffmpeg, битые файлы).
# Пишем через soundfile, а НЕ через torchaudio.save: в этой версии
# torchaudio.save требует пакет torchcodec и падает с ImportError.
import numpy as np
import soundfile as sf

def generate_synthetic_stems(num_tracks=12, num_valid=2, duration_sec=6,
                             sample_rate=44100, seed=0):
    """Создаёт синтетические треки в формате проекта."""
    rng = np.random.default_rng(seed)
    samples = int(duration_sec * sample_rate)
    t = np.arange(samples, dtype="float32") / sample_rate

    print(f"Генерирую {num_tracks} синтетических треков ({duration_sec} сек)...")
    for i in range(num_tracks):
        split = "valid" if i >= num_tracks - num_valid else "train"
        out_dir = os.path.join(OUT_ROOT, split, f"track_{i + 1:03d}")
        if os.path.isdir(out_dir):
            continue
        os.makedirs(out_dir, exist_ok=True)

        # 4 «инструмента»: разные частоты + шум, чтобы не было
        # вырожденного случая, где все дорожки одинаковые.
        freqs = [220.0, 55.0, 110.0, 440.0]
        amps = [0.30, 0.45, 0.35, 0.18]
        stems = {}
        for name, freq, amp in zip(SOURCES, freqs, amps):
            wave = np.sin(2 * np.pi * freq * t).astype("float32")
            # лёгкая частотная модуляция — звучит живее, чем чистый синус
            wave *= (0.8 + 0.2 * np.sin(2 * np.pi * 0.7 * t)).astype("float32")
            wave += rng.standard_normal(samples).astype("float32") * 0.05
            mono = (wave * amp).astype("float32")
            stems[name] = np.stack([mono, mono])   # моно → стерео

        mixture = np.clip(sum(stems.values()), -1.0, 1.0).astype("float32")
        sf.write(os.path.join(out_dir, "mixture.wav"), mixture.T,
                 sample_rate, subtype="PCM_16")
        for name, arr in stems.items():
            sf.write(os.path.join(out_dir, f"{name}.wav"), arr.T,
                     sample_rate, subtype="PCM_16")

    print(f"✅ Синтетические треки готовы в {OUT_ROOT}")

counts = {split: len(glob.glob(os.path.join(OUT_ROOT, split, "track_*")))
          for split in ("train", "valid")}

if counts["train"] < 4 or counts["valid"] < 1:
    print("\n⚠️  Реальных треков мало (train=%d, valid=%d) — генерирую синтетику"
          % (counts["train"], counts["valid"]))
    generate_synthetic_stems()
    counts = {split: len(glob.glob(os.path.join(OUT_ROOT, split, "track_*")))
              for split in ("train", "valid")}
    synthetic = True
else:
    print(f"\n✅ Используем реальные треки из сэмпла: train={counts['train']}, valid={counts['valid']}")
    print("Ячейку с синтетикой можно пропустить.")
    synthetic = False

print(f"\nДатасет: train={counts['train']}, valid={counts['valid']}")

In [ ]:
# Запуск быстрого обучения для проверки пайплайна
from config import Config
from train_custom import train_custom_model

config = Config()
config.device = "cuda"
config.custom_data_path = OUT_ROOT
config.output_model_path = MODEL_PATH
config.output_dir = "/content/output"
config.batch_size = 2
config.epochs = 5
config.segment_length = 44100 * 6   # 6 сек: треки в сэмпле по 6.8 сек
config.num_workers = 2
config.checkpoint_interval = 5
config.early_stopping = False      # на 5 эпохах останавливаться рано
config.save_checkpoints = True

problems = config.validate()
if problems:
    for item in problems:
        print("❌", item)
    raise ValueError("Конфиг не прошёл проверку")

print("=" * 50)
print("🧪 БЫСТРАЯ ПРОВЕРКА ПАЙПЛАЙНА")
print("=" * 50)
print(f"GPU           : {torch.cuda.get_device_name(0)}")
print(f"Эпох          : {config.epochs}")
print(f"Batch size    : {config.batch_size}")
print(f"Длина куска   : {config.segment_length / config.sample_rate:.1f} сек")
print(f"Данные        : {'синтетика' if synthetic else '7s сэмпл (реальные треки)'}")
print(f"Треков        : {counts['train']} train / {counts['valid']} valid")
print(f"Модель        : {config.output_model_path}")
print("=" * 50)

# assume_yes=True — критично для Colab: внутри есть input() с вопросом
# «продолжить с чекпоинта?». Без stdin он ловит EOFError, и обучение
# молча начинается с нуля при каждом перезапуске ноутбука.
best_model_path = train_custom_model(config, assume_yes=True)

print(f"\n🎉 ПАЙПЛАЙН РАБОТАЕТ!")
print(f"Модель сохранена: {best_model_path}")
print(f"Чекпоинт: {str(best_model_path).replace('.pth', '_last.pth')}")

if os.path.isfile(best_model_path):
    size_mb = os.path.getsize(best_model_path) / 1e6
    print(f"✅ Файл на диске: {size_mb:.1f} МБ")

# 🎉 Проверка пайплайна завершена!

## Если видишь "🎉 ПАЙПЛАЙН РАБОТАЕТ!" — значит работает:
- Обучение на GPU
- Чтение данных (DataLoader, 4 дорожки)
- Прямой и обратный проход, подсчёт потери
- Сохранение модели в Google Drive
- Сохранение чекпоинта для продолжения

## Что смотреть в логе
- **потеря падает** от эпохи к эпохе — модель учится
- если потеря на 1-й эпохе уже очень маленькая — треки слишком простые
  (для чистых синусов из резервной ячейки это ожидаемо)
- время эпохи: при 7s-треках это секунды, а не минуты

## Про качество модели
Модель, обученная на 7-секундных отрезках, **не годится** для продакшена:
слишком мало данных и слишком короткие куски. Она нужна только чтобы
проверить, что код не падает. Для нормальной работы нужны полные треки.

## Следующий шаг: полный MUSDB18
Официально 7s-сэмпл — это всё, что отдаёт пакет `musdb`.
Полный набор (100 треков, 4-5 минут каждый) лежит на musdb.io
и требует регистрации — автоматической загрузки нет.

Практичный путь без регистрации — **собрать датасет из своих треков**:
1. Собери папку с любой музыкой (mp3/flac/wav)
2. Пропусти через `convert_audio.py` — получишь нормализованный wav
3. Разложи по структуре `prepare_data_structure.py --help`
4. Запусти `colab_training.ipynb` (v1) — там логика для полных треков

## Если Colab отключился
Чекпоинт в Drive, обучение продолжится с последней сохранённой эпохи